# 🏙️ Jeddah Property Investment Scorecard
### Vision 2030-Aligned Real Estate Evaluation Tool

**Author:** [Your Name]  
**Data Sources:** REGA, JLL KSA Residential Market Review, Cavendish Maxwell H1 2025, GAStat  
**Last Updated:** 2026

---

This notebook evaluates residential properties in Jeddah across four dimensions:
1. **Rental Yield** — vs. district and city benchmarks  
2. **Price Benchmarking** — price/sqm vs. district averages  
3. **Vision 2030 Location Score** — proximity to strategic development zones  
4. **Supply Risk** — based on new unit pipeline by district  

Outputs a composite **Investment Score (0–100)** with a Buy / Hold / Avoid verdict.


## 1. Install & Import Libraries

In [ ]:
# Install required libraries (Colab only)
!pip install plotly --quiet

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries loaded successfully')

## 2. Reference Data
Market benchmarks sourced from JLL KSA Living Market Dynamics (Q2 2025), Cavendish Maxwell H1 2025 Report, and GAStat Real Estate Price Index.

In [ ]:
# ─────────────────────────────────────────────
# DISTRICT REFERENCE DATA
# Sources: Cavendish Maxwell H1 2025, JLL Q2 2025
# Prices in SAR/sqm | Yields in %
# ─────────────────────────────────────────────

DISTRICT_DATA = {
    # District: [apt_price_sqm, villa_price_sqm, avg_yield_pct, vision2030_score, supply_risk_score]
    # vision2030_score: 1-10 (10 = highest strategic alignment)
    # supply_risk_score: 1-10 (10 = highest risk from new supply)

    'Al-Shati (Corniche)':        [5800, 7200, 7.8, 9.5, 4.0],
    'Al-Zahra':                   [5200, 6500, 7.5, 8.5, 3.5],
    'Al-Nahdah':                  [4800, 6000, 7.6, 8.0, 4.5],
    'Obhur Al-Shamaliyah':        [5500, 7000, 7.2, 8.8, 7.0],  # High new supply
    'Al-Murjan':                  [5300, 6800, 7.4, 8.6, 7.5],  # High new supply
    'Al-Basateen':                [4600, 5800, 8.0, 7.5, 6.5],
    'Jeddah Central Zone':        [5600, 7100, 8.2, 10.0, 5.0], # Jeddah Central project
    'Al-Hamra':                   [5100, 6400, 7.9, 8.2, 3.0],
    'Al-Rawdah':                  [4400, 5600, 8.1, 7.0, 2.5],
    'Al-Salamah':                 [4200, 5300, 8.3, 6.5, 2.0],
    'Al-Balad (Historic)':        [3800, 4900, 6.5, 7.8, 3.5],  # Heritage zone
    'Al-Faisaliyah':              [4000, 5100, 8.5, 6.0, 2.0],
    'Mishrifah':                  [3900, 5000, 8.6, 5.5, 2.5],
    'Al-Marwah':                  [4300, 5500, 8.2, 6.8, 3.0],
    'Al-Khalidiyah':              [4700, 5900, 7.7, 7.2, 3.5],
}

# City-wide benchmarks (Cavendish Maxwell H1 2025)
CITY_BENCHMARKS = {
    'apt_price_sqm':    4376,   # SAR/sqm city average
    'villa_price_sqm':  5114,   # SAR/sqm city average
    'avg_yield':        7.89,   # % gross rental yield
    'apt_yoy_growth':   1.8,    # % year-on-year price growth
    'villa_yoy_growth': 2.5,    # % year-on-year price growth
    'rental_yoy_apts':  4.7,    # % rental rate growth
}

# Convert to DataFrame for easy display
df_districts = pd.DataFrame.from_dict(
    DISTRICT_DATA, orient='index',
    columns=['Apt SAR/sqm', 'Villa SAR/sqm', 'Avg Yield %', 'Vision2030 Score', 'Supply Risk']
).reset_index().rename(columns={'index': 'District'})

print('📊 District Reference Table')
print('─' * 75)
print(df_districts.to_string(index=False))
print(f'\n🏙️ Jeddah City Average: SAR {CITY_BENCHMARKS["apt_price_sqm"]:,}/sqm (Apts) | SAR {CITY_BENCHMARKS["villa_price_sqm"]:,}/sqm (Villas)')
print(f'📈 City Average Gross Rental Yield: {CITY_BENCHMARKS["avg_yield"]}%')

## 3. Scoring Engine

In [ ]:
# ─────────────────────────────────────────────
# SCORING ENGINE
# ─────────────────────────────────────────────

WEIGHTS = {
    'rental_yield':   0.40,   # 40% — income return
    'price_value':    0.25,   # 25% — price vs benchmark
    'vision2030':     0.20,   # 20% — strategic location
    'supply_risk':    0.15,   # 15% — downside protection
}

def calculate_gross_yield(monthly_rent_sar, purchase_price_sar):
    """Calculate annual gross rental yield."""
    return (monthly_rent_sar * 12 / purchase_price_sar) * 100

def score_rental_yield(yield_pct, district_avg_yield):
    """Score yield relative to district average. Max 100."""
    ratio = yield_pct / district_avg_yield
    if ratio >= 1.15:    return 100
    elif ratio >= 1.05:  return 85
    elif ratio >= 0.95:  return 65
    elif ratio >= 0.85:  return 45
    else:                return 25

def score_price_value(price_sqm, district_benchmark_sqm, property_type):
    """Score price/sqm vs district benchmark. Below benchmark = good value."""
    ratio = price_sqm / district_benchmark_sqm
    if ratio <= 0.90:    return 100  # >10% below benchmark — strong value
    elif ratio <= 0.97:  return 80   # Slight discount
    elif ratio <= 1.03:  return 60   # At market
    elif ratio <= 1.10:  return 40   # Slight premium
    else:                return 20   # >10% premium

def score_supply_risk(supply_risk_raw):
    """Invert supply risk — lower risk = higher score."""
    return round((10 - supply_risk_raw) * 10, 1)  # Returns 0-100

def score_vision2030(vision_score_raw):
    """Normalize vision2030 score to 0-100."""
    return vision_score_raw * 10  # Already 1-10

def get_verdict(composite_score):
    """Investment verdict based on composite score."""
    if composite_score >= 72:
        return ('BUY', '🟢', 'Strong investment fundamentals. Yield above district average, favourable price positioning, and Vision 2030 tailwinds support capital appreciation.')
    elif composite_score >= 52:
        return ('HOLD', '🟡', 'Adequate fundamentals with mixed signals. Consider negotiating price or waiting for a stronger entry point. Monitor supply pipeline.')
    else:
        return ('AVOID', '🔴', 'Weak risk-adjusted return. Price premium or below-average yield reduces margin of safety. Explore alternative districts.')

def evaluate_property(district, property_type, size_sqm, asking_price_sar, monthly_rent_sar):
    """
    Main evaluation function.
    Returns dict of all scores and verdict.
    """
    if district not in DISTRICT_DATA:
        raise ValueError(f'District not found. Choose from: {list(DISTRICT_DATA.keys())}')

    d = DISTRICT_DATA[district]
    apt_bench, villa_bench, avg_yield, v2030, supply_risk = d

    bench_sqm = apt_bench if property_type.lower() == 'apartment' else villa_bench
    input_price_sqm = asking_price_sar / size_sqm

    # Calculate metrics
    gross_yield = calculate_gross_yield(monthly_rent_sar, asking_price_sar)
    annual_rent  = monthly_rent_sar * 12

    # Raw scores (0-100)
    s_yield   = score_rental_yield(gross_yield, avg_yield)
    s_price   = score_price_value(input_price_sqm, bench_sqm, property_type)
    s_supply  = score_supply_risk(supply_risk)
    s_vision  = score_vision2030(v2030)

    # Weighted composite
    composite = (
        s_yield  * WEIGHTS['rental_yield'] +
        s_price  * WEIGHTS['price_value']  +
        s_vision * WEIGHTS['vision2030']   +
        s_supply * WEIGHTS['supply_risk']
    )

    verdict, icon, rationale = get_verdict(composite)

    return {
        'district':          district,
        'property_type':     property_type,
        'size_sqm':          size_sqm,
        'asking_price_sar':  asking_price_sar,
        'monthly_rent_sar':  monthly_rent_sar,
        'price_per_sqm':     round(input_price_sqm, 0),
        'district_bench_sqm': bench_sqm,
        'price_vs_bench_pct': round((input_price_sqm / bench_sqm - 1) * 100, 1),
        'gross_yield_pct':   round(gross_yield, 2),
        'district_avg_yield': avg_yield,
        'annual_rent_sar':   annual_rent,
        'score_yield':       s_yield,
        'score_price':       s_price,
        'score_vision2030':  s_vision,
        'score_supply':      s_supply,
        'composite_score':   round(composite, 1),
        'verdict':           verdict,
        'verdict_icon':      icon,
        'rationale':         rationale,
        'vision2030_raw':    v2030,
        'supply_risk_raw':   supply_risk,
    }

print('✅ Scoring engine loaded')
print(f'📐 Weights: Yield {WEIGHTS["rental_yield"]*100}% | Price {WEIGHTS["price_value"]*100}% | Vision2030 {WEIGHTS["vision2030"]*100}% | Supply Risk {WEIGHTS["supply_risk"]*100}%')

## 4. Visualization Functions

In [ ]:
# ─────────────────────────────────────────────
# VISUALIZATION FUNCTIONS
# ─────────────────────────────────────────────

def plot_investment_scorecard(result):
    """4-panel scorecard dashboard."""

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[
            'Composite Investment Score',
            'Score Breakdown by Dimension',
            'Price/sqm vs District Benchmark',
            'Rental Yield vs District & City Average'
        ],
        specs=[[{'type': 'indicator'}, {'type': 'bar'}],
               [{'type': 'bar'},       {'type': 'bar'}]]
    )

    score_color = '#2ecc71' if result['composite_score'] >= 72 else ('#f39c12' if result['composite_score'] >= 52 else '#e74c3c')

    # Panel 1 — Gauge
    fig.add_trace(go.Indicator(
        mode='gauge+number+delta',
        value=result['composite_score'],
        title={'text': f"{result['verdict_icon']} {result['verdict']}", 'font': {'size': 18}},
        gauge={
            'axis': {'range': [0, 100]},
            'bar': {'color': score_color},
            'steps': [
                {'range': [0, 52],  'color': '#fadbd8'},
                {'range': [52, 72], 'color': '#fdebd0'},
                {'range': [72, 100],'color': '#d5f5e3'},
            ],
            'threshold': {'line': {'color': 'black', 'width': 3}, 'thickness': 0.75, 'value': result['composite_score']}
        }
    ), row=1, col=1)

    # Panel 2 — Score breakdown
    dims   = ['Rental Yield', 'Price Value', 'Vision 2030', 'Supply Safety']
    scores = [result['score_yield'], result['score_price'], result['score_vision2030'], result['score_supply']]
    colors = ['#3498db' if s >= 65 else '#e67e22' if s >= 45 else '#e74c3c' for s in scores]

    fig.add_trace(go.Bar(
        x=dims, y=scores, marker_color=colors,
        text=[f'{s:.0f}' for s in scores], textposition='outside',
        showlegend=False
    ), row=1, col=2)

    # Panel 3 — Price/sqm comparison
    city_bench = CITY_BENCHMARKS['apt_price_sqm'] if result['property_type'].lower() == 'apartment' else CITY_BENCHMARKS['villa_price_sqm']
    fig.add_trace(go.Bar(
        x=['This Property', 'District Avg', 'City Avg'],
        y=[result['price_per_sqm'], result['district_bench_sqm'], city_bench],
        marker_color=['#9b59b6', '#3498db', '#95a5a6'],
        text=[f'SAR {v:,.0f}' for v in [result['price_per_sqm'], result['district_bench_sqm'], city_bench]],
        textposition='outside', showlegend=False
    ), row=2, col=1)

    # Panel 4 — Yield comparison
    fig.add_trace(go.Bar(
        x=['This Property', 'District Avg', 'City Avg'],
        y=[result['gross_yield_pct'], result['district_avg_yield'], CITY_BENCHMARKS['avg_yield']],
        marker_color=['#2ecc71', '#3498db', '#95a5a6'],
        text=[f"{v:.1f}%" for v in [result['gross_yield_pct'], result['district_avg_yield'], CITY_BENCHMARKS['avg_yield']]],
        textposition='outside', showlegend=False
    ), row=2, col=2)

    fig.update_layout(
        height=650,
        title_text=f"🏙️ Investment Scorecard — {result['district']} | {result['property_type'].title()} | SAR {result['asking_price_sar']:,}",
        title_font_size=16,
        showlegend=False,
        plot_bgcolor='white',
        paper_bgcolor='#f8f9fa'
    )
    fig.update_yaxes(range=[0, 115], row=1, col=2)
    fig.update_yaxes(row=2, col=2, ticksuffix='%')
    fig.show()


def plot_district_comparison(property_type='apartment'):
    """Compare all districts on yield and Vision 2030 score (bubble chart)."""
    districts, yields, v2030_scores, prices, supply = [], [], [], [], []

    for dist, vals in DISTRICT_DATA.items():
        districts.append(dist)
        prices.append(vals[0] if property_type == 'apartment' else vals[1])
        yields.append(vals[2])
        v2030_scores.append(vals[3])
        supply.append(vals[4])

    fig = px.scatter(
        x=yields, y=v2030_scores,
        size=prices,
        color=supply,
        color_continuous_scale='RdYlGn_r',
        text=districts,
        labels={'x': 'Average Gross Yield (%)', 'y': 'Vision 2030 Strategic Score', 'color': 'Supply Risk'},
        title='Jeddah District Matrix — Yield vs Vision 2030 Alignment (bubble size = price/sqm)',
        height=550
    )
    fig.update_traces(textposition='top center', textfont_size=9)
    fig.update_layout(
        plot_bgcolor='white',
        paper_bgcolor='#f8f9fa',
        xaxis=dict(gridcolor='#eeeeee'),
        yaxis=dict(gridcolor='#eeeeee')
    )
    # Add quadrant lines
    fig.add_hline(y=7.5, line_dash='dash', line_color='#aaaaaa', opacity=0.5)
    fig.add_vline(x=7.9, line_dash='dash', line_color='#aaaaaa', opacity=0.5)
    fig.add_annotation(x=8.5, y=9.8, text='⭐ High Yield + High V2030', showarrow=False, font=dict(color='green', size=10))
    fig.show()

print('✅ Visualization functions loaded')

## 5. Run Evaluation — Edit Your Property Here

In [ ]:
# ─────────────────────────────────────────────
#  ✏️  EDIT YOUR PROPERTY DETAILS BELOW
# ─────────────────────────────────────────────

PROPERTY = {
    'district':         'Al-Zahra',          # See district list above
    'property_type':    'apartment',          # 'apartment' or 'villa'
    'size_sqm':         120,                  # Size in sqm
    'asking_price_sar': 750_000,              # Purchase price in SAR
    'monthly_rent_sar': 5_200,                # Expected monthly rent in SAR
}

# ─── RUN EVALUATION ───────────────────────────
result = evaluate_property(**PROPERTY)

# ─── PRINT SUMMARY ────────────────────────────
print('=' * 60)
print('  JEDDAH PROPERTY INVESTMENT SCORECARD')
print('=' * 60)
print(f"  District       : {result['district']}")
print(f"  Type           : {result['property_type'].title()}")
print(f"  Size           : {result['size_sqm']} sqm")
print(f"  Asking Price   : SAR {result['asking_price_sar']:,}")
print(f"  Monthly Rent   : SAR {result['monthly_rent_sar']:,}")
print('-' * 60)
print(f"  Price/sqm      : SAR {result['price_per_sqm']:,.0f}  (District avg: SAR {result['district_bench_sqm']:,})")
print(f"  vs Benchmark   : {result['price_vs_bench_pct']:+.1f}%")
print(f"  Gross Yield    : {result['gross_yield_pct']:.2f}%  (District avg: {result['district_avg_yield']}%)")
print(f"  Annual Rent    : SAR {result['annual_rent_sar']:,}")
print('-' * 60)
print(f"  Yield Score    : {result['score_yield']}/100")
print(f"  Price Score    : {result['score_price']}/100")
print(f"  Vision 2030    : {result['score_vision2030']}/100")
print(f"  Supply Safety  : {result['score_supply']}/100")
print('=' * 60)
print(f"  COMPOSITE SCORE: {result['composite_score']}/100")
print(f"  VERDICT        : {result['verdict_icon']} {result['verdict']}")
print('-' * 60)
print(f"  {result['rationale']}")
print('=' * 60)

In [ ]:
# Plot the scorecard dashboard
plot_investment_scorecard(result)

## 6. District Market Overview

In [ ]:
# Full district comparison bubble chart
plot_district_comparison(property_type='apartment')

## 7. Batch Compare Multiple Properties

In [ ]:
# ─────────────────────────────────────────────
#  Compare multiple properties side by side
# ─────────────────────────────────────────────

PORTFOLIO = [
    {'district': 'Al-Zahra',             'property_type': 'apartment', 'size_sqm': 120, 'asking_price_sar': 750_000,   'monthly_rent_sar': 5_200},
    {'district': 'Jeddah Central Zone',  'property_type': 'apartment', 'size_sqm': 95,  'asking_price_sar': 680_000,   'monthly_rent_sar': 4_900},
    {'district': 'Al-Rawdah',            'property_type': 'villa',     'size_sqm': 350, 'asking_price_sar': 2_100_000, 'monthly_rent_sar': 15_500},
    {'district': 'Obhur Al-Shamaliyah',  'property_type': 'apartment', 'size_sqm': 140, 'asking_price_sar': 900_000,   'monthly_rent_sar': 5_800},
    {'district': 'Al-Shati (Corniche)',  'property_type': 'apartment', 'size_sqm': 110, 'asking_price_sar': 850_000,   'monthly_rent_sar': 5_500},
]

results = [evaluate_property(**p) for p in PORTFOLIO]

df_compare = pd.DataFrame([{
    'District':         r['district'],
    'Type':             r['property_type'].title(),
    'Price (SAR)':      f"{r['asking_price_sar']:,}",
    'Yield %':          f"{r['gross_yield_pct']:.2f}%",
    'vs Benchmark':     f"{r['price_vs_bench_pct']:+.1f}%",
    'Score':            r['composite_score'],
    'Verdict':          f"{r['verdict_icon']} {r['verdict']}"
} for r in results])

df_compare_sorted = df_compare.sort_values('Score', ascending=False)

print('📊 Portfolio Comparison — Ranked by Investment Score')
print('─' * 85)
print(df_compare_sorted.to_string(index=False))

# Bar chart
fig = go.Figure(go.Bar(
    y=[r['district'] for r in sorted(results, key=lambda x: x['composite_score'])],
    x=[r['composite_score'] for r in sorted(results, key=lambda x: x['composite_score'])],
    orientation='h',
    marker_color=['#e74c3c' if r['composite_score'] < 52 else '#f39c12' if r['composite_score'] < 72 else '#2ecc71'
                  for r in sorted(results, key=lambda x: x['composite_score'])],
    text=[f"{r['verdict_icon']} {r['composite_score']}" for r in sorted(results, key=lambda x: x['composite_score'])],
    textposition='outside'
))
fig.add_vline(x=72, line_dash='dash', line_color='green', annotation_text='BUY threshold')
fig.add_vline(x=52, line_dash='dash', line_color='orange', annotation_text='HOLD threshold')
fig.update_layout(
    title='Portfolio Comparison — Investment Scores',
    xaxis_title='Investment Score (0–100)',
    xaxis=dict(range=[0, 115]),
    height=350,
    plot_bgcolor='white',
    paper_bgcolor='#f8f9fa'
)
fig.show()

## 8. Market Context Summary
Key data points for analyst reference.

In [ ]:
market_context = """
╔══════════════════════════════════════════════════════════════════╗
║         JEDDAH REAL ESTATE MARKET CONTEXT — H1 2025             ║
╠══════════════════════════════════════════════════════════════════╣
║  Market Size      SAR 18.3B in sales (+34% YoY) H1 2025         ║
║  Transactions     15,200 units (+25% YoY) — Cavendish Maxwell    ║
║  Apt Prices       SAR 4,376/sqm (+1.8% YoY)                     ║
║  Villa Prices     SAR 5,114/sqm (+2.5% YoY)                     ║
║  Apt Rent Growth  +4.7% YoY                                     ║
║  Villa Rent       -2.7% YoY (oversupply pressure)               ║
║  Gross Yield      ~7.89% city average (JLL / STC RE Index)      ║
║  New Units        ~12,700 units pipeline by end-2025             ║
║  Supply Hotspots  Obhur Al-Shamaliyah, Al-Murjan, Al-Basateen   ║
║  Key Projects     Jeddah Central ($20B), Jeddah Tower ($7.2B)   ║
║  Foreign Law      Non-Saudis can own in designated zones (2026)  ║
╚══════════════════════════════════════════════════════════════════╝
Sources: Cavendish Maxwell H1 2025 | JLL KSA Living Q2 2025 |
         GAStat RE Price Index | Mordor Intelligence Jan 2026
"""
print(market_context)